In [3]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 4, Finished, Available, Finished, False)

In [4]:
bronze_df = spark.table("bronze_netflix_titles")

#display(bronze_df)

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 5, Finished, Available, Finished, False)

In [5]:
silver_df = bronze_df.dropDuplicates(["show_id"])

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 6, Finished, Available, Finished, False)

In [6]:
print("Bronze :", bronze_df.count())
print("Silver :", silver_df.count())

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 7, Finished, Available, Finished, False)

Bronze : 8806
Silver : 8806


In [7]:
invalid_rows = silver_df.filter(
    ~col("show_id").startswith("s")
)
display(invalid_rows)

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 53817726-c8f6-4593-9136-eb4c402c61ab)

In [8]:
#removing spaces in front and back
string_columns = [
    field.name
    for field in silver_df.schema.fields
    if isinstance(field.dataType, StringType)
]

for c in string_columns:
    silver_df = silver_df.withColumn(
        c,
        trim(col(c))
    )

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 9, Finished, Available, Finished, False)

In [9]:
#Find all columns that hold text (StringType)
string_columns = [
    field.name 
    for field in silver_df.schema.fields 
    if isinstance(field.dataType, StringType)
]

# 2. Loop through only those text columns and strip the double quotes
for c in string_columns:
    silver_df = silver_df.withColumn(c, regexp_replace(col(c), '"', ''))


StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 10, Finished, Available, Finished, False)

In [10]:
#filling all the null values
silver_df = silver_df.fillna({
    "director": "Unknown",
    "cast": "Unknown",
    "country": "Unknown",
    "rating": "Not Rated",
    "duration": "Unknown"
})

silver_df = silver_df.filter(~col("show_id").isin("s2211", "s1769", "s7031", "s6210", "s74", "s199", "s6840", "s6574", "s7032", "s733", "s1412", "s1416", "s3813", "s2674", "s5892", "s171", "s1252", "s2138"))
#display(silver_df.orderBy(col("release_year").desc()))
#display(silver_df)
#silver_df.createOrReplaceTempView("silver_layer")

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 11, Finished, Available, Finished, False)

In [11]:
#converting date format

silver_df = silver_df.withColumn(
    "date_added",
    to_date(
        col("date_added"),
        "MMMM d, yyyy"
    )
)

silver_df.select("date_added").show(5)

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 12, Finished, Available, Finished, False)

+----------+
|date_added|
+----------+
|2021-09-25|
|2021-09-24|
|2021-09-07|
|2021-04-22|
|2021-04-22|
+----------+
only showing top 5 rows



In [12]:
silver_df = silver_df.withColumn(
    "content_age",
    year(current_date()) - col("release_year")
)

silver_df = silver_df.withColumn(
    "added_year",
    year(col("date_added"))
)

silver_df = silver_df.withColumn(
    "added_month",
    month(col("date_added"))
)

silver_df = silver_df.withColumn(
    "added_month_name",
    date_format(
        col("date_added"),
        "MMMM"
    )
)

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 13, Finished, Available, Finished, False)

In [13]:
silver_df = silver_df.withColumn(
    "genre_array",
    split(col("listed_in"), ",")
)

silver_df = silver_df.withColumn(
    "genre_array",
    expr("""
        transform(
            genre_array,
            x -> trim(x)
        )
    """)
)

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 14, Finished, Available, Finished, False)

In [14]:
silver_df = silver_df.withColumn(
    "country_array",
    split(col("country"), ",")
)

silver_df = silver_df.withColumn(
    "country_array",
    expr("""
        transform(
            country_array,
            x -> trim(x)
        )
    """)
)

#display(silver_df.limit(30))

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 15, Finished, Available, Finished, False)

In [15]:
silver_df = silver_df.withColumn(
    "silver_processed_time",
    current_timestamp()
)

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 16, Finished, Available, Finished, False)

In [18]:
spark.sql("DROP TABLE IF EXISTS silver_netflix_titles")

(
    silver_df.write
        .mode("overwrite")
        .format("delta")
        .saveAsTable("silver_netflix_titles")
)

display(
    spark.table("silver_netflix_titles")
)

StatementMeta(, 2e6ea3f4-fb3c-455a-b6ad-dc15600fad06, 19, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8e076662-dd0a-4d29-ab40-4953ef2c05ed)